In [1]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git
!pip install -q datasets gluonts transformers accelerate scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.4 MB/s eta 0:00:00


In [14]:
import sys
sys.path.insert(0, "/content/chronos-forecasting/src")
import torch
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from chronos import ChronosConfig, ChronosPipeline
from transformers import AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from torch.utils.data import IterableDataset
from datasets import load_dataset
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [39]:
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 48
MAX_STEPS = 500
BATCH_SIZE = 8
MAX_TRAIN_SERIES = 200
MAX_TEST_SERIES = 500

FINE_TUNE_DATASETS = [
    ("m4_hourly", "M4 Hourly", "Economics"),
    ("monash_electricity_hourly", "Electricity", "Energy"),
    ("monash_traffic", "Traffic", "Transport"),
]

ALL_TEST_DATASETS = [
    ("m4_yearly", "M4 Yearly", "Economics"),
    ("m4_quarterly", "M4 Quarterly", "Economics"),
    ("m4_monthly", "M4 Monthly", "Economics"),
    ("m4_weekly", "M4 Weekly", "Economics"),
    ("m4_daily", "M4 Daily", "Economics"),
    ("m4_hourly", "M4 Hourly", "Economics"),
    ("monash_electricity_hourly", "Electricity", "Energy"),
    ("monash_traffic", "Traffic", "Transport"),
    ("monash_weather", "Weather", "Meteorology"),
    ("monash_nn5_weekly", "NN5 Weekly", "Retail"),
]

In [40]:
def load_and_prepare_dataset(dataset_name, context_len, pred_len, max_series=None):
    dataset = load_dataset("autogluon/chronos_datasets", dataset_name, split="train")
    df = dataset.to_pandas()

    series_ids = df['id'].unique()

    all_series = []

    for series_id in series_ids:
        series = df[df['id'] == series_id]['target'].values[0]
        if isinstance(series, np.ndarray):
            series = series.tolist()
        if len(series) > context_len + pred_len + 10:
            series = series[:context_len + pred_len]
            all_series.append(series)

    if len(all_series) == 0:
        return [], []

    if max_series is not None and len(all_series) > max_series:
        all_series = all_series[:max_series]

    train_series, test_series = train_test_split(
        all_series,
        test_size=0.2,
        random_state=42
    )

    return train_series, test_series

def prepare_data(series_list, context_len, pred_len):
    data = []
    for series in series_list:
        if len(series) < context_len + pred_len:
            continue
        for i in range(len(series) - context_len - pred_len + 1):
            context = series[i:i+context_len]
            future = series[i+context_len:i+context_len+pred_len]
            if len(context) == context_len and len(future) == pred_len:
                data.append({
                    "context": context,
                    "future": future,
                })
    return data

In [41]:
class ChronosDataset(IterableDataset):
    def __init__(self, data, tokenizer, config):
        self.data = data
        self.tokenizer = tokenizer
        self.vocab_size = config.n_tokens + config.n_special_tokens
        self.indices = list(range(len(data)))

    def __iter__(self):
        np.random.shuffle(self.indices)
        for idx in self.indices:
            item = self.data[idx]
            context = torch.tensor(item["context"], dtype=torch.float32).unsqueeze(0)
            future = torch.tensor(item["future"], dtype=torch.float32).unsqueeze(0)

            input_ids, attention_mask, scale = self.tokenizer.context_input_transform(context)
            labels, _ = self.tokenizer.label_input_transform(future, scale)
            labels = torch.clamp(labels, 0, self.vocab_size - 1)

            yield {
                "input_ids": input_ids.squeeze(0),
                "attention_mask": attention_mask.squeeze(0),
                "labels": labels.squeeze(0),
            }

In [42]:
def evaluate_model(pipeline, test_series, pred_len):
    results = []
    for series in test_series:
        if len(series) < pred_len:
            continue
        context = series[:-pred_len]
        actual = series[-pred_len:]

        with torch.no_grad():
            forecast = pipeline.predict(
                torch.tensor(context, dtype=torch.float32),
                prediction_length=pred_len,
                num_samples=20
            )
            forecast_np = forecast.numpy()
            if forecast_np.ndim == 3:
                forecast_np = forecast_np[0]
            median = np.median(forecast_np, axis=0)

        results.append({
            'mae': mean_absolute_error(actual, median),
        })

    return np.mean([r['mae'] for r in results]) if results else None

In [43]:
pipeline_zs = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=torch.float32,
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [44]:
import time

fine_tuned_models = {}
zero_shot_results_all = {}

for ds_name, ds_display, ds_domain in ALL_TEST_DATASETS:
    _, test_series = load_and_prepare_dataset(
        ds_name, CONTEXT_LENGTH, PREDICTION_LENGTH, max_series=MAX_TEST_SERIES
    )
    if test_series:
        avg_mae = evaluate_model(pipeline_zs, test_series, PREDICTION_LENGTH)
        zero_shot_results_all[ds_display] = avg_mae
        print(f"   {ds_display}: {avg_mae:.3f} ({len(test_series)} рядов)")
    else:
        print(f"   {ds_display}: error")

finetuned_results_all = {}

for ds_name, ds_display, ds_domain in FINE_TUNE_DATASETS:
    print(f"{ds_display} ({ds_domain})")

    train_series, test_series = load_and_prepare_dataset(
        ds_name, CONTEXT_LENGTH, PREDICTION_LENGTH, max_series=MAX_TRAIN_SERIES
    )

    print(f"   Обучающих рядов: {len(train_series)}")
    print(f"   Тестовых рядов: {len(test_series)}")

    train_data = prepare_data(train_series, CONTEXT_LENGTH, PREDICTION_LENGTH)
    print(f"   Создано {len(train_data)} примеров")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        "amazon/chronos-t5-small",
        torch_dtype=torch.float32,
    )
    model = model.to(device)

    config = ChronosConfig(
        tokenizer_class="MeanScaleUniformBins",
        tokenizer_kwargs={"low_limit": -15.0, "high_limit": 15.0},
        n_tokens=4096,
        pad_token_id=0,
        eos_token_id=1,
        use_eos_token=True,
        model_type="seq2seq",
        context_length=CONTEXT_LENGTH,
        prediction_length=PREDICTION_LENGTH,
        n_special_tokens=2,
        num_samples=20,
        temperature=1.0,
        top_k=50,
        top_p=1.0,
    )
    tokenizer = config.create_tokenizer()

    train_dataset = ChronosDataset(train_data, tokenizer, config)

    output_dir = f"./finetuned_{ds_name}/"
    output_dir_path = Path(output_dir)
    output_dir_path.mkdir(parents=True, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir_path),
        per_device_train_batch_size=BATCH_SIZE,
        learning_rate=1e-5,
        max_steps=MAX_STEPS,
        logging_steps=100,
        save_steps=500,
        save_total_limit=2,
        optim="adamw_torch",
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        remove_unused_columns=False,
        dataloader_num_workers=0,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    trainer.train()

    final_checkpoint_dir = output_dir_path / "checkpoint-final"
    trainer.save_model(str(final_checkpoint_dir))

    pipeline_ft = ChronosPipeline.from_pretrained(
        str(final_checkpoint_dir),
        device_map=device,
        torch_dtype=torch.float32,
    )

    finetuned_results_all[ds_display] = {}

    for test_ds_name, test_ds_display, test_ds_domain in ALL_TEST_DATASETS:
        _, test_series = load_and_prepare_dataset(
            test_ds_name, CONTEXT_LENGTH, PREDICTION_LENGTH, max_series=MAX_TEST_SERIES
        )
        if test_series:
            avg_mae = evaluate_model(pipeline_ft, test_series, PREDICTION_LENGTH)
            finetuned_results_all[ds_display][test_ds_display] = avg_mae
            zs = zero_shot_results_all.get(test_ds_display, 0)
            imp = ((zs - avg_mae) / zs * 100) if zs > 0 else 0
            print(f"      {test_ds_display}: {avg_mae:.3f} ({imp:+.1f}%) ({len(test_series)} рядов)")
        else:
            print(f"      {test_ds_display}: error")

    fine_tuned_models[ds_display] = pipeline_ft

    del model, trainer, pipeline_ft
    torch.cuda.empty_cache()
    time.sleep(2)

   M4 Yearly: 2389.435 (2 рядов)
   M4 Quarterly: 344.082 (9 рядов)
   M4 Monthly: 532.740 (100 рядов)


m4_weekly/train-00000-of-00001.parquet:   0%|          | 0.00/2.56M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/359 [00:00<?, ? examples/s]

   M4 Weekly: 506.754 (50 рядов)


m4_daily/train-00000-of-00001.parquet:   0%|          | 0.00/65.5M [00:00<?, ?B/s]

...

Step,Training Loss
100,3.955710
200,3.819596
300,3.740882
400,3.684693
500,3.657729


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

      M4 Yearly: 2404.619 (-0.6%) (2 рядов)
      M4 Quarterly: 382.999 (-11.3%) (9 рядов)
      M4 Monthly: 528.452 (+0.8%) (100 рядов)
      M4 Weekly: 546.152 (-7.8%) (50 рядов)
      M4 Daily: 273.972 (-0.8%) (100 рядов)
      M4 Hourly: 252.257 (+0.5%) (83 рядов)
      Electricity: 206.307 (-7.4%) (65 рядов)
      Traffic: 0.009 (+26.2%) (100 рядов)
      Weather: 1.978 (-0.1%) (100 рядов)
      NN5 Weekly: error


In [45]:
all_comparison = []
for test_ds_display in zero_shot_results_all.keys():
    zs = zero_shot_results_all.get(test_ds_display, 0)
    row = {'Dataset': test_ds_display, 'Zero-shot': round(zs, 3)}
    for ft_ds in finetuned_results_all.keys():
        ft_val = finetuned_results_all[ft_ds].get(test_ds_display, None)
        if ft_val is not None:
            imp = ((zs - ft_val) / zs * 100) if zs > 0 else 0
            row[f'{ft_ds} (FT)'] = f"{ft_val:.3f} ({imp:+.1f}%)"
        else:
            row[f'{ft_ds} (FT)'] = "—"
    all_comparison.append(row)

comparison_df = pd.DataFrame(all_comparison)
print(comparison_df.to_string(index=False))

     Dataset  Zero-shot   M4 Hourly (FT) Electricity (FT)     Traffic (FT)
   M4 Yearly   2389.435 2390.303 (-0.0%) 2383.131 (+0.3%) 2404.619 (-0.6%)
M4 Quarterly    344.082 389.324 (-13.1%) 388.697 (-13.0%) 382.999 (-11.3%)
  M4 Monthly    532.740  537.128 (-0.8%)  545.687 (-2.4%)  528.452 (+0.8%)
   M4 Weekly    506.754  496.813 (+2.0%)  499.279 (+1.5%)  546.152 (-7.8%)
    M4 Daily    271.786  266.366 (+2.0%)  266.164 (+2.1%)  273.972 (-0.8%)
   M4 Hourly    253.533  256.156 (-1.0%)  261.228 (-3.0%)  252.257 (+0.5%)
 Electricity    192.072  188.172 (+2.0%)  178.898 (+6.9%)  206.307 (-7.4%)
     Traffic      0.013    0.013 (-1.4%)    0.013 (-5.4%)   0.009 (+26.2%)
     Weather      1.976    1.984 (-0.4%)    1.984 (-0.4%)    1.978 (-0.1%)
